# 🤖 AI Ethics Education LLM - Training Notebook

**Llama 3.1 8B + QLoRA Fine-tuning for AI Ethics Education**

This notebook trains a specialized AI ethics assistant covering:
- EU AI Act
- OECD AI Principles  
- Partnership on AI
- UNESCO Ethical Guidelines

---

## 📋 Requirements
- **GPU**: T4 (free) or better (A100 recommended)
- **Runtime**: GPU runtime
- **Time**: ~2-3 hours on T4, ~1 hour on A100
- **Hugging Face Account**: Required for Llama 3.1 access

---

## 🚀 Quick Start
1. **Runtime → Change runtime type → GPU (T4 or A100)**
2. Run cells in order
3. Enter your Hugging Face token when prompted
4. Wait for training to complete
5. Test with Gradio demo at the end!

## 1️⃣ Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n{'='*60}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Training will be very slow.")
print(f"{'='*60}")

In [ ]:
%%time
# Install required packages
print("📦 Installing dependencies...\n")

!pip install -q torch transformers>=4.40.0 datasets>=2.18.0 accelerate>=0.26.0
!pip install -q peft>=0.10.0 bitsandbytes>=0.43.0 gradio>=4.0.0
!pip install -q pandas requests beautifulsoup4 tqdm

print("\n✅ Installation complete!")

## 2️⃣ Clone Project Repository

In [ ]:
import os

# Clone the repository
if not os.path.exists('ethicsedullm'):
    !git clone https://github.com/Educatian/ethicsedullm.git
    print("✅ Repository cloned!")
else:
    print("ℹ️ Repository already exists")

# Change to project directory
%cd ethicsedullm

# List files
!ls -lh *.py | head -15

## 3️⃣ Hugging Face Setup

**Required:** Get Llama 3.1 access
1. Go to: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
2. Click "Accept License" 
3. Get your token: https://huggingface.co/settings/tokens
4. Enter it below

In [ ]:
from huggingface_hub import login
from getpass import getpass

# Option 1: Manual token input (recommended for security)
hf_token = getpass("Enter your Hugging Face token: ")
login(token=hf_token)

# Option 2: Uncomment to use Colab secrets (if configured)
# from google.colab import userdata
# hf_token = userdata.get('HF_TOKEN')
# login(token=hf_token)

print("\n✅ Logged in to Hugging Face!")

## 4️⃣ Prepare Training Data

In [ ]:
%%time
# Generate training dataset
print("📚 Generating training dataset...\n")
!python data_collection.py

# Verify dataset
print("\n" + "="*60)
!ls -lh ai_ethics_dataset.jsonl
!wc -l ai_ethics_dataset.jsonl

In [ ]:
# Preview dataset
import json
from collections import Counter

with open('ai_ethics_dataset.jsonl', 'r') as f:
    entries = [json.loads(line) for line in f]

print(f"📊 Dataset Statistics")
print("="*60)
print(f"Total examples: {len(entries)}")
print(f"\nCategories: {dict(Counter(e.get('category', 'N/A') for e in entries))}")
print(f"\nSample entry:")
print(f"Q: {entries[0]['instruction']}")
print(f"A: {entries[0]['response'][:200]}...")
print("="*60)

## 5️⃣ Model Training (QLoRA)

**Training Configuration:**
- Base Model: Llama 3.1 8B Instruct
- Method: QLoRA (4-bit quantization)
- LoRA rank: 16
- Epochs: 3
- Batch size: 4 (with gradient accumulation)

**Expected Time:**
- T4 GPU: ~2-3 hours
- A100 GPU: ~1 hour

⚠️ **Don't close this tab during training!**

In [ ]:
%%time
# Train the model
print("🚀 Starting training...\n")
print("This will take 1-3 hours depending on your GPU.")
print("You'll see training progress below.\n")
print("="*60)

!python model_training_qlora.py

## 6️⃣ Model Evaluation

In [ ]:
%%time
# Evaluate the trained model
print("📊 Evaluating model performance...\n")
!python evaluation_rubric.py

In [ ]:
# Show evaluation results
import json

if os.path.exists('evaluation_report.json'):
    with open('evaluation_report.json', 'r') as f:
        report = json.load(f)
    
    print("\n" + "="*70)
    print("📊 EVALUATION RESULTS")
    print("="*70)
    
    summary = report['summary']
    print(f"\nOverall Scores:")
    print(f"  Combined Score:  {summary['average_combined_score']:.3f}")
    print(f"  Keyword Score:   {summary['average_keyword_score']:.3f}")
    print(f"  Rubric Score:    {summary['average_rubric_score']:.3f}")
    
    print(f"\nRubric Breakdown:")
    for dim, data in report['rubric_breakdown'].items():
        print(f"  {dim:20s}: {data['average']:.3f} (weight: {data['weight']:.2f})")
    
    print("\n" + "="*70)
    
    # Show a sample response
    if report['detailed_results']:
        sample = report['detailed_results'][0]
        print("\n📝 Sample Response:")
        print(f"Q: {sample['question']}")
        print(f"\nA: {sample['response'][:300]}...")
        print(f"\nScore: {sample['evaluation']['combined_score']:.3f}")
else:
    print("⚠️ Evaluation report not found. Run evaluation first.")

## 7️⃣ Interactive Demo (Gradio)

Test your trained model with a web interface!

In [ ]:
# Launch Gradio demo
print("🌐 Launching Gradio demo...\n")
print("The demo will open in a new browser tab.")
print("You can ask questions about AI ethics!\n")

!python app.py

### Quick Test (No UI)

In [ ]:
# Quick test without Gradio
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from prompt_templates import format_llama_prompt

print("Loading model...")

# Load model with quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "./ai_ethics_llm_final")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

print("✅ Model loaded!\n")

# Test questions
test_questions = [
    "What is algorithmic bias?",
    "What are the key principles of the EU AI Act?",
    "How can organizations implement ethical AI governance?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"[{i}] Q: {question}")
    print(f"{'='*70}\n")
    
    prompt = format_llama_prompt(question, include_system=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            inputs.input_ids,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    
    if "<|start_header_id|>assistant<|end_header_id|>" in response:
        response = response.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    
    print(f"A: {response.strip()}\n")

## 8️⃣ Save & Upload to Hugging Face (Optional)

In [ ]:
# Optional: Merge LoRA weights for easier deployment
print("🔄 Merging LoRA weights with base model...\n")
!python merge_lora.py

print("\n✅ Merged model saved to ./ai_ethics_llm_merged/")

In [ ]:
# Upload to Hugging Face Hub (replace with your username)
from huggingface_hub import HfApi

# Configuration
REPO_NAME = "your-username/ai-ethics-llm-8b"  # Change this!
MODEL_PATH = "./ai_ethics_llm_merged"  # or "./ai_ethics_llm_final" for LoRA adapters

print(f"📤 Uploading model to {REPO_NAME}...\n")
print("⚠️ Make sure to change REPO_NAME to your username!\n")

# Uncomment to upload:
# api = HfApi()
# api.create_repo(repo_id=REPO_NAME, exist_ok=True, private=False)
# api.upload_folder(
#     folder_path=MODEL_PATH,
#     repo_id=REPO_NAME,
#     repo_type="model"
# )
# print(f"\n✅ Model uploaded to https://huggingface.co/{REPO_NAME}")

print("\nUncomment the code above to upload.")

## 9️⃣ Download Model Files

In [ ]:
# Create a zip file for download
!zip -r ai_ethics_llm_final.zip ai_ethics_llm_final/

print("\n✅ Model zipped!")
print("\nTo download:")
print("1. Click the folder icon on the left sidebar")
print("2. Find 'ai_ethics_llm_final.zip'")
print("3. Right-click → Download")

# Show file size
!ls -lh ai_ethics_llm_final.zip

---

## ✅ Summary

You've successfully:
1. ✅ Set up GPU environment
2. ✅ Prepared AI ethics training data (30 examples)
3. ✅ Fine-tuned Llama 3.1 8B with QLoRA
4. ✅ Evaluated the model with hybrid metrics
5. ✅ Tested with Gradio demo

### 📁 Generated Files:
- `ai_ethics_dataset.jsonl` - Training data
- `ai_ethics_llm_final/` - LoRA adapters
- `ai_ethics_llm_merged/` - Full merged model (if created)
- `evaluation_report.json` - Evaluation results

### 🚀 Next Steps:
1. Download the model (see above)
2. Upload to Hugging Face Hub
3. Deploy with Gradio/Streamlit
4. Fine-tune on your own data

### 📚 Resources:
- GitHub: https://github.com/Educatian/ethicsedullm
- Llama 3.1: https://ai.meta.com/llama/
- QLoRA Paper: https://arxiv.org/abs/2305.14314

---

**Questions or issues?** Open an issue on GitHub!